In [1]:
import control as ctrl
import numpy as np
import matplotlib.pyplot as plt

# Problema de levitador magnético 

Definimos el sistema en estudio

In [2]:
G=ctrl.tf(-40,[1,20,-100,-2000])
G

TransferFunction(
array([-40]),
array([    1,    20,  -100, -2000]),
outputs=1, inputs=1)

In [3]:
G.pole()

AttributeError: 'TransferFunction' object has no attribute 'pole'

In [ ]:
s=ctrl.tf('s')

Otra forma de definirlo:

In [ ]:
Gs=-40/((s+20)*(s**2-100))
Gs

Lo analizamos con el bode

In [ ]:
ctrl.bode(Gs, dB=True)
plt.gcf().set_size_inches(12,8)

In [ ]:
zeta=0.46
sv=np.exp(-np.pi*zeta/(np.sqrt(1-zeta**2)))
sv

Obtenermos el $\omega_n$

In [ ]:
wn=4/0.4/0.46
wn

Del resutlado anterior defino $\omega_c=22$, calculo la ganancia necesaria para obtener esa frecuencia de corte

In [ ]:
wc=22
k1=1/np.abs(G(wc*1j))
k1

In [ ]:
ctrl.stability_margins(G*k1)

In [ ]:
T=ctrl.feedback(k1*G)
T.pole()

Vemos que hicimos un controlador inestable. Vamos a estudiar la estabildad con Nyquist (en la pizarra). Luego verificamos con el Nyquist.

In [ ]:
ctrl.rlocus(-G);
plt.gcf().set_size_inches(8,6)

In [ ]:
ctrl.rlocus(G);
plt.gcf().set_size_inches(8,6)

Vemos que pasa en la frecuencia de cruce que necesitamos

In [ ]:
ctrl.bode(-k1*G);
plt.gcf().set_size_inches(12,8)

In [ ]:
ctrl.stability_margins(-k1*G)

Podemos ver que el sistema neceista más de 90 grados lo cual es imposible de conseguir con un solo cero. Vamos a tener que agregar un cero y un lead como mínimo. 

Vamos a tratar de agregar un cero, en algún lugar que no molesta al transitorio pero que agrega fase. Este lugar podria ser -10, tapando el polo real negativo de la planta.

In [ ]:
d1=(s+10)

In [ ]:
ctrl.bode(-k1*G*d1);
plt.gcf().set_size_inches(12,8)

In [ ]:
phi_max=50*np.pi/180
alpha = (1-np.sin(phi_max))/(1+np.sin(phi_max))
alpha

In [ ]:
wmax=22
TD=1/(wmax*np.sqrt(alpha))
z=-1/TD
p=-1/(alpha*TD)
z, p

In [ ]:
d2=(-s/z+1)/(-s/p+1)
d2

In [ ]:
kd2=1/np.abs(d2(wc*1j))
ctrl.bode(d2*kd2, dB=True);
plt.gcf().set_size_inches(12,8)

In [ ]:
C1=d1*d2*kd2
ctrl.bode(-G*d1*d2*kd2, dB=True);
fig=plt.gcf()
fig.set_size_inches(12,8)

In [ ]:
k3=1/np.abs((-G*d1*d2*kd2)(22j))
k3

In [ ]:
T=ctrl.feedback(-G*d1*d2*kd2*k3)
T.pole()

Vemos que este diseno no resultó del todo correcto. Este se debe a varias razones:
- poca ganancia en estado estacionario
- un polo negativo chico que resulta dominante frente a los que diseñamos

In [ ]:
t,y=ctrl.step_response(T, T=np.linspace(0,5,2000))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t,y)
ax.set_title('Respuesta al escalón')
ax.set_xlabel('Tiempo')
ax.grid()

In [ ]:
L=-G*d1*d2*kd2*k3
ctrl.bode(L, dB=True);

In [ ]:
fig=plt.gcf()
fig.set_size_inches(12,8)

In [ ]:
G

In [ ]:
G.pole()

Para lograr un controlador que en principio funcione y epnsar de otra manera los lead-lag podemos hacer loop-shaping, y después ajusta a nuestro a lo que necesitemos.

In [ ]:
Cls=((s+10)*(s+10)*(s+20))/s

In [ ]:
ctrl.bode(-G*Cls);
fig=plt.gcf()
fig.set_size_inches(12,8)

Analizamos que margen de fase tenemos a 22 rad y luego vemos la respuesta. Podemos después redisenar y mejorar aumentando la frecuencia de corte. Recordar que los isstemas inestables son más faciles de controlar más rápido que lento

In [ ]:
kls=1/np.abs((G*Cls)(wc*1j))

In [ ]:
ctrl.bode(-G*Cls*kls);
fig=plt.gcf()
fig.set_size_inches(12,8)

In [ ]:
ctrl.stability_margins(-G*Cls*kls)

In [ ]:
T=ctrl.feedback(-G*Cls*kls)
t,y=ctrl.step_response(T)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t,y)
ax.set_title('Respuesta al escalón')
ax.set_xlabel('Tiempo')
ax.grid()

In [ ]:
Cls

In [ ]:
Cls2=Cls*1/(s/400+1)**2
Cls2

In [ ]:
Tu=ctrl.feedback(-Cls2,G)
t,y=ctrl.step_response(Tu, T=np.linspace(0,1,2000))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t,y)
ax.set_title('Respuesta al escalón')
ax.set_xlabel('Tiempo')
ax.grid()